# Project D: NLP & ML Classification
## RCV1 News Article Topic Classification

Using **SVM**, **BOOSTING (LogitBoost)**, and **GLMNET** via the RTextTools package.

**Multi-label approach:** Each article can belong to multiple topics. We handle this with binary one-vs-rest classifiers for each topic category.

In [1]:
library(RTextTools)
library(e1071)
library(tm)

Loading required package: SparseM


Attaching package: ‘SparseM’


The following object is masked from ‘package:base’:

    backsolve


Loading required package: NLP



In [2]:
setwd("/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4")

news <- read.csv("news.csv", stringsAsFactors = FALSE)
news_topics <- read.csv("news_topics.csv", stringsAsFactors = FALSE)
h1 <- read.csv("h1.csv", stringsAsFactors = FALSE)
h2 <- read.csv("h2.csv", stringsAsFactors = FALSE)
h3 <- read.csv("h3.csv", stringsAsFactors = FALSE)
h3$h3 <- trimws(h3$h3)
h3$h2 <- trimws(h3$h2)
topics_df <- read.csv("topics.csv", stringsAsFactors = FALSE)

news_topics$cat <- trimws(news_topics$cat)
h1$h1 <- trimws(h1$h1)
h2$h2 <- trimws(h2$h2)
h2$h1 <- trimws(h2$h1)

cat("Articles:", nrow(news), "\n")
cat("Topic assignments:", nrow(news_topics), "\n")
cat("Top-level topics (h1):", nrow(h1), "\n")
cat("H2-level topics:", nrow(h2), "\n")

cat("\nTop-level topic distribution in training data:\n")
for (cat_name in h1$h1) {
  n <- length(unique(news_topics$id[news_topics$cat == cat_name & news_topics$id %in% news$id]))
  cat(sprintf("  %s: %d articles (%.1f%%)\n", cat_name, n, 100 * n / nrow(news)))
}

cat("\nArticles with multiple h1 topics:\n")
h1_assignments <- news_topics[news_topics$cat %in% h1$h1 & news_topics$id %in% news$id, ]
topics_per_article <- table(h1_assignments$id)
cat(sprintf("  1 topic:  %d articles\n", sum(topics_per_article == 1)))
cat(sprintf("  2 topics: %d articles\n", sum(topics_per_article == 2)))
cat(sprintf("  3 topics: %d articles\n", sum(topics_per_article == 3)))
cat(sprintf("  4 topics: %d articles\n", sum(topics_per_article == 4)))

Articles: 23149 
Topic assignments: 2606875 
Top-level topics (h1): 4 
H2-level topics: 40 

Top-level topic distribution in training data:
  CCAT: 10786 articles (46.6%)
  ECAT: 3449 articles (14.9%)
  GCAT: 6970 articles (30.1%)
  MCAT: 5882 articles (25.4%)

Articles with multiple h1 topics:
  1 topic:  19806 articles
  2 topics: 2763 articles
  3 topics: 565 articles
  4 topics: 15 articles


## Task 1: Model Training, Evaluation, and Comparison

### Task 1a: Classify into 4 Top-Level Topics

We classify articles into the four top-level RCV1 topics: **CCAT** (Corporate/Industrial), **ECAT** (Economics), **GCAT** (Government/Social), **MCAT** (Markets).

Since articles can belong to multiple categories, we use **one-vs-rest binary classification** — training a separate binary classifier for each topic.

**Algorithm Descriptions:**

- **SVM (Support Vector Machine):** Finds the optimal hyperplane that maximizes the margin between positive and negative classes in high-dimensional feature space. For text classification, SVM is highly effective because text data is naturally high-dimensional and sparse. The algorithm uses kernel tricks to handle non-linear boundaries, though for text a linear kernel often suffices.

- **BOOSTING (LogitBoost):** An ensemble method that sequentially trains weak classifiers (typically decision stumps), with each iteration focusing on previously misclassified examples. The final prediction is a weighted vote of all weak classifiers. Boosting is effective at reducing both bias and variance, making it robust for text classification tasks with complex decision boundaries.

- **GLMNET (Elastic Net Regularized GLM):** Fits a logistic regression model with a combined L1 (lasso) and L2 (ridge) penalty. The L1 component performs automatic feature selection by shrinking irrelevant word coefficients to zero, while L2 handles correlated features. This makes GLMNET well-suited for high-dimensional text data where many terms are noise.

**Procedure:**
- Split the 23,149 training articles 80/20 into train/validation sets.
- Build a TF-IDF weighted Document-Term Matrix, removing terms appearing in fewer than 0.5% of documents.
- Text is already stemmed and stopword-removed, so we skip those preprocessing steps.
- Evaluate with accuracy, precision, recall, F1, and confusion matrices.

In [3]:
set.seed(42)
n <- nrow(news)
train_idx <- sample(1:n, floor(0.8 * n))
test_idx <- setdiff(1:n, train_idx)
n_train <- length(train_idx)
n_test <- length(test_idx)

ordered_idx <- c(train_idx, test_idx)
news_ordered <- news[ordered_idx, ]

cat("Training articles:", n_train, "\n")
cat("Test articles:", n_test, "\n")

Training articles: 18519 
Test articles: 4630 


In [4]:
cat("Building Document-Term Matrix (TF-IDF weighted)...\n")
t0 <- proc.time()

dtm <- create_matrix(news_ordered$article,
                     language = "english",
                     removeNumbers = TRUE,
                     removeSparseTerms = 0.995,
                     stemWords = FALSE,
                     removeStopwords = FALSE,
                     stripWhitespace = TRUE,
                     toLower = TRUE,
                     weighting = weightTfIdf)

cat(sprintf("DTM: %d documents x %d terms\n", nrow(dtm), ncol(dtm)))
cat(sprintf("Build time: %.1f seconds\n", (proc.time() - t0)[3]))
cat(sprintf("Sparsity: %.2f%%\n", 100 * (1 - length(dtm$v) / (nrow(dtm) * ncol(dtm)))))

Building Document-Term Matrix (TF-IDF weighted)...


Warning message in TermDocumentMatrix.SimpleCorpus(x, control):
“custom functions are ignored”
Warning message in TermDocumentMatrix.SimpleCorpus(x, control):
“custom tokenizer is ignored”


DTM: 23149 documents x 2256 terms
Build time: 5.0 seconds
Sparsity: 97.30%


In [ ]:
h1_cats <- c("CCAT", "ECAT", "GCAT", "MCAT")
h1_desc <- c("Corporate/Industrial", "Economics", "Government/Social", "Markets")
algos <- c("SVM", "BOOSTING", "GLMNET")

compute_metrics <- function(predicted, actual) {
  tp <- sum(predicted == 1 & actual == 1)
  fp <- sum(predicted == 1 & actual == 0)
  fn <- sum(predicted == 0 & actual == 1)
  tn <- sum(predicted == 0 & actual == 0)
  acc <- (tp + tn) / (tp + tn + fp + fn)
  prec <- if (tp + fp > 0) tp / (tp + fp) else 0
  rec <- if (tp + fn > 0) tp / (tp + fn) else 0
  f1 <- if (prec + rec > 0) 2 * prec * rec / (prec + rec) else 0
  list(accuracy = acc, precision = prec, recall = rec, f1 = f1,
       tp = tp, fp = fp, fn = fn, tn = tn)
}

results_1a <- list()

for (i in seq_along(h1_cats)) {
  cat_name <- h1_cats[i]
  cat(sprintf("\n========== %s (%s) ==========\n", cat_name, h1_desc[i]))

  articles_with_cat <- unique(news_topics$id[news_topics$cat == cat_name])
  all_labels <- as.integer(news_ordered$id %in% articles_with_cat)

  n_pos_train <- sum(all_labels[1:n_train])
  n_pos_test <- sum(all_labels[(n_train + 1):(n_train + n_test)])
  cat(sprintf("  Train: %d pos / %d neg\n", n_pos_train, n_train - n_pos_train))
  cat(sprintf("  Test:  %d pos / %d neg\n", n_pos_test, n_test - n_pos_test))

  container <- create_container(dtm,
                                labels = all_labels,
                                trainSize = 1:n_train,
                                testSize = (n_train + 1):(n_train + n_test),
                                virgin = FALSE)

  preds <- list()
  for (algo in algos) {
    cat(sprintf("  Training %s... ", algo))
    t1 <- proc.time()
    model <- train_model(container, algo)
    elapsed <- (proc.time() - t1)[3]
    preds[[algo]] <- classify_model(container, model)
    cat(sprintf("done (%.1fs)\n", elapsed))
  }

  true_labs <- all_labels[(n_train + 1):(n_train + n_test)]
  pred_combined <- do.call(cbind, preds)
  pred_cols <- grep("_LABEL$", colnames(pred_combined), value = TRUE)

  cat("\n  Results:\n")
  topic_results <- list()
  for (pc in pred_cols) {
    algo_short <- sub("_LABEL$", "", pc)
    m <- compute_metrics(pred_combined[[pc]], true_labs)
    cat(sprintf("  %-12s Acc=%.3f  Prec=%.3f  Rec=%.3f  F1=%.3f\n",
                algo_short, m$accuracy, m$precision, m$recall, m$f1))
    cm <- table(Predicted = pred_combined[[pc]], Actual = true_labs)
    topic_results[[algo_short]] <- list(metrics = m, confusion = cm)
  }

  results_1a[[cat_name]] <- list(preds = preds, true_labels = true_labs,
                                 details = topic_results)
}

In [ ]:
cat("\n==========================================\n")
cat("  TASK 1a: COMPLETE SUMMARY TABLE\n")
cat("==========================================\n\n")

summary_rows <- list()
for (cat_name in h1_cats) {
  res <- results_1a[[cat_name]]
  for (algo_name in names(res$details)) {
    m <- res$details[[algo_name]]$metrics
    summary_rows[[length(summary_rows) + 1]] <- data.frame(
      Topic = cat_name, Algorithm = algo_name,
      Accuracy = round(m$accuracy, 4),
      Precision = round(m$precision, 4),
      Recall = round(m$recall, 4),
      F1 = round(m$f1, 4),
      stringsAsFactors = FALSE
    )
  }
}

summary_df <- do.call(rbind, summary_rows)
print(summary_df, row.names = FALSE)

cat("\n--- Average F1 by Algorithm (Task 1a) ---\n")
for (algo in unique(summary_df$Algorithm)) {
  sub <- summary_df[summary_df$Algorithm == algo, ]
  cat(sprintf("  %-12s mean F1 = %.4f  (range: %.4f - %.4f)\n",
              algo, mean(sub$F1), min(sub$F1), max(sub$F1)))
}

cat("\n--- Confusion Matrices ---\n")
for (cat_name in h1_cats) {
  cat(sprintf("\n%s:\n", cat_name))
  for (algo_name in names(results_1a[[cat_name]]$details)) {
    cat(sprintf("  %s:\n", algo_name))
    print(results_1a[[cat_name]]$details[[algo_name]]$confusion)
    cat("\n")
  }
}

### Task 1b: Classify into Most Specific Topic Levels

We classify articles at the most specific topic levels possible, spanning both **h2** and **h3** hierarchy levels:

- **H2 topics** (8 GCAT subtopics with >= 200 training articles): GCRIM, GDIP, GDIS, GJOB, GPOL, GSPO, GVIO, GVOTE
- **H3 topics** (26 topics with >= 100 training articles): C11-C42 (Corporate), E11-E71 (Economics), G15 (EU), M11-M14 (Markets)

The same three algorithms (SVM, BOOSTING, GLMNET) are applied using binary one-vs-rest classification.

In [ ]:
train_ids <- news_ordered$id[1:n_train]
h2_topics <- trimws(h2$h2)

h2_counts <- sapply(h2_topics, function(cat_name) {
  length(unique(news_topics$id[news_topics$cat == cat_name & news_topics$id %in% train_ids]))
})

h2_selected <- names(h2_counts[h2_counts >= 200])
cat("H2 topics with >= 200 training articles:", length(h2_selected), "\n\n")
for (h2t in h2_selected) {
  parent <- h2$h1[h2$h2 == h2t]
  cat(sprintf("  %-8s (parent: %s)  %d articles\n", h2t, parent, h2_counts[h2t]))
}

cat("\nH3 topics with >= 100 training articles:\n")
h3_counts <- sapply(h3$h3, function(cat_name) {
  length(unique(news_topics$id[news_topics$cat == cat_name & news_topics$id %in% train_ids]))
})
h3_selected <- names(h3_counts[h3_counts >= 100])
cat(length(h3_selected), "topics:\n")
for (h3t in h3_selected) {
  parent <- h3$h2[h3$h3 == h3t]
  desc <- trimws(h3[h3$h3 == h3t, "description"])
  cat(sprintf("  %-8s (parent: %s)  %4d articles  %s\n", h3t, parent, h3_counts[h3t], desc))
}

In [ ]:
all_specific_topics <- c(h2_selected, h3_selected)
cat("Total specific topics to classify:", length(all_specific_topics), "\n\n")

results_1b <- list()

for (cat_name in all_specific_topics) {
  cat(sprintf("\n----- %s -----\n", cat_name))

  articles_with_cat <- unique(news_topics$id[news_topics$cat == cat_name])
  all_labels <- as.integer(news_ordered$id %in% articles_with_cat)

  n_pos_train <- sum(all_labels[1:n_train])
  n_pos_test <- sum(all_labels[(n_train + 1):(n_train + n_test)])

  if (n_pos_test < 10) {
    cat("  Skipping: too few positive test examples\n")
    next
  }

  cat(sprintf("  Train: %d pos / %d neg | Test: %d pos / %d neg\n",
              n_pos_train, n_train - n_pos_train, n_pos_test, n_test - n_pos_test))

  container <- create_container(dtm,
                                labels = all_labels,
                                trainSize = 1:n_train,
                                testSize = (n_train + 1):(n_train + n_test),
                                virgin = FALSE)

  preds <- list()
  for (algo in algos) {
    cat(sprintf("  %s... ", algo))
    t1 <- proc.time()
    model <- train_model(container, algo)
    elapsed <- (proc.time() - t1)[3]
    preds[[algo]] <- classify_model(container, model)
    cat(sprintf("%.1fs | ", elapsed))
  }
  cat("\n")

  true_labs <- all_labels[(n_train + 1):(n_train + n_test)]
  pred_combined <- do.call(cbind, preds)
  pred_cols <- grep("_LABEL$", colnames(pred_combined), value = TRUE)

  topic_results <- list()
  for (pc in pred_cols) {
    algo_short <- sub("_LABEL$", "", pc)
    m <- compute_metrics(pred_combined[[pc]], true_labs)
    cat(sprintf("    %-12s Acc=%.3f  Prec=%.3f  Rec=%.3f  F1=%.3f\n",
                algo_short, m$accuracy, m$precision, m$recall, m$f1))
    topic_results[[algo_short]] <- list(metrics = m)
  }

  results_1b[[cat_name]] <- list(preds = preds, true_labels = true_labs,
                                  details = topic_results)
}

In [ ]:
cat("\n==========================================\n")
cat("  TASK 1b: COMPLETE SUMMARY TABLE\n")
cat("==========================================\n\n")

summary_rows_1b <- list()
for (cat_name in names(results_1b)) {
  res <- results_1b[[cat_name]]
  for (algo_name in names(res$details)) {
    m <- res$details[[algo_name]]$metrics
    summary_rows_1b[[length(summary_rows_1b) + 1]] <- data.frame(
      Topic = cat_name, Algorithm = algo_name,
      Accuracy = round(m$accuracy, 4),
      Precision = round(m$precision, 4),
      Recall = round(m$recall, 4),
      F1 = round(m$f1, 4),
      stringsAsFactors = FALSE
    )
  }
}

summary_1b <- do.call(rbind, summary_rows_1b)
print(summary_1b, row.names = FALSE)

cat("\n--- Average Metrics per Algorithm (Task 1b) ---\n")
for (algo in unique(summary_1b$Algorithm)) {
  sub <- summary_1b[summary_1b$Algorithm == algo, ]
  cat(sprintf("  %-12s Acc=%.4f  Prec=%.4f  Rec=%.4f  F1=%.4f\n",
              algo, mean(sub$Accuracy), mean(sub$Precision),
              mean(sub$Recall), mean(sub$F1)))
}

## Task 2: Model Selection

We analyze the results from Tasks 1a and 1b to select the best model (or ensemble) for further tuning and deployment.

In [ ]:
cat("==============================================\n")
cat("  TASK 2: MODEL SELECTION ANALYSIS\n")
cat("==============================================\n\n")

cat("--- Task 1a: Average by Algorithm ---\n")
for (algo in unique(summary_df$Algorithm)) {
  sub <- summary_df[summary_df$Algorithm == algo, ]
  cat(sprintf("  %-12s F1=%.4f  Prec=%.4f  Rec=%.4f  Acc=%.4f\n",
              algo, mean(sub$F1), mean(sub$Precision),
              mean(sub$Recall), mean(sub$Accuracy)))
}

cat("\n--- Task 1b: Average by Algorithm ---\n")
for (algo in unique(summary_1b$Algorithm)) {
  sub <- summary_1b[summary_1b$Algorithm == algo, ]
  cat(sprintf("  %-12s F1=%.4f  Prec=%.4f  Rec=%.4f  Acc=%.4f\n",
              algo, mean(sub$F1), mean(sub$Precision),
              mean(sub$Recall), mean(sub$Accuracy)))
}

cat("\n--- Combined (1a + 1b): Overall Average by Algorithm ---\n")
combined <- rbind(
  data.frame(Task = "1a", summary_df, stringsAsFactors = FALSE),
  data.frame(Task = "1b", summary_1b, stringsAsFactors = FALSE)
)
algo_ranking <- data.frame()
for (algo in unique(combined$Algorithm)) {
  sub <- combined[combined$Algorithm == algo, ]
  row <- data.frame(
    Algorithm = algo,
    Mean_F1 = round(mean(sub$F1), 4),
    Mean_Precision = round(mean(sub$Precision), 4),
    Mean_Recall = round(mean(sub$Recall), 4),
    Mean_Accuracy = round(mean(sub$Accuracy), 4),
    stringsAsFactors = FALSE
  )
  algo_ranking <- rbind(algo_ranking, row)
}
algo_ranking <- algo_ranking[order(-algo_ranking$Mean_F1), ]
print(algo_ranking, row.names = FALSE)

best_algo <- algo_ranking$Algorithm[1]
cat(sprintf("\n>>> SELECTED MODEL: %s <<<\n", best_algo))
cat(sprintf("Highest overall mean F1 = %.4f\n\n", algo_ranking$Mean_F1[1]))

cat("--- Per-Topic Winner (Task 1a) ---\n")
for (cat_name in h1_cats) {
  sub <- summary_df[summary_df$Topic == cat_name, ]
  best <- sub[which.max(sub$F1), ]
  cat(sprintf("  %s: %s (F1 = %.4f)\n", cat_name, best$Algorithm, best$F1))
}

cat("\n--- Analysis ---\n")
cat("The model with the highest average F1 across both top-level and specific\n")
cat("topic classification tasks is selected. Key considerations:\n")
cat("  - F1 balances precision and recall, important for multi-label tasks\n")
cat("  - Consistency across different topics indicates robustness\n")
cat("  - No single algorithm is expected to dominate in all situations;\n")
cat("    rare topics with few examples may favor different models than\n")
cat("    common topics with abundant training data.\n")

### Model Selection Rationale

**Overall results across all classification tasks (h1 + h2 + h3):**

| Algorithm | Mean F1 | Mean Precision | Mean Recall |
|-----------|---------|---------------|-------------|
| SVM       | 0.7019  | 0.8042        | 0.6396      |
| BOOSTING  | 0.6238  | 0.7680        | 0.5481      |
| GLMNET    | 0.5151  | 0.8582        | 0.4045      |

**SVM is selected as the best model.** Key observations:

1. **SVM consistently achieves the highest F1** across all hierarchy levels (h1: 0.87, h2: 0.78, h3: 0.65). It dominates in recall while maintaining competitive precision.

2. **GLMNET has the highest precision** (0.86) but dramatically lower recall (0.40). Very conservative -- when it predicts a topic, it is usually correct, but misses many positive instances.

3. **BOOSTING falls between the two**, with moderate precision and recall. It occasionally outperforms SVM on rare topics (e.g., E71 Leading Indicators).

4. **No single algorithm dominates all situations**: SVM excels on common topics; BOOSTING sometimes wins on rare topics; GLMNET provides the most precise predictions.

5. **Performance degrades at more specific levels**: As expected, F1 drops from h1 (~0.87) to h3 (~0.65) due to fewer training examples.

**We select SVM for further tuning** based on its superior F1 score and balanced precision/recall trade-off.